# Fundamentals of Machine Learning — Exercise 5

## Gradient Descent, Linear Regression, and Logistic Regression

### 🎯 Aim of the Exercise

In this exercise, you will:

- understand how **gradient descent** minimizes a loss function (and do one step by hand),
- implement gradient descent for **linear regression from scratch** and test it on synthetic and real data,
- investigate the effect of the **learning rate**,
- apply the same optimization idea to **logistic regression**,
- evaluate a binary classifier using a **confusion matrix, accuracy, precision, recall, F1-score, ROC curve and AUC**,
- see how the **decision threshold** changes the errors a classifier makes.


## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import make_regression, load_breast_cancer, load_diabetes
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_curve,
    roc_auc_score,
    mean_absolute_error,
    mean_squared_error,
)

np.random.seed(42)

## 2. Gradient Descent

Gradient descent is an iterative optimization algorithm used to minimize a **loss function**.

*Intuition:* you stand on a hill in fog. You cannot see the valley, but you can feel the slope under your feet. You take a step downhill, feel the slope again, and repeat.

At each iteration, we:

1. use the current parameters to make predictions,
2. calculate the loss and its gradient,
3. update the parameters in the opposite direction of the gradient.

$$
\theta_{new} = \theta_{old} - \alpha \nabla L(\theta)
$$

- $\theta$ represents the model parameters,
- $\alpha$ is the **learning rate** (the length of the step),
- $\nabla L(\theta)$ is the gradient of the loss (the direction of steepest *increase*).

❓ Why do we move in the opposite direction of the gradient?

❓ What might happen if the learning rate is too small or too large?

### Gradient descent step by step

Tiny dataset: $x = (0,1,2,3)$, $y = (1,3,5,7)$ (the true line is $y = 2x+1$). Start with $w=0,\ b=0$ and use $\alpha = 0.05$.

The model is $\hat y = wx+b$ and the loss is $MSE=\frac1n\sum(y_i-\hat y_i)^2$. The gradients are

$$
\frac{\partial MSE}{\partial w}=\frac{2}{n}\sum_i(\hat y_i-y_i)\,x_i,
\qquad
\frac{\partial MSE}{\partial b}=\frac{2}{n}\sum_i(\hat y_i-y_i).
$$

Run the cell below. Each line is one gradient-descent step. Watch how $w$, $b$, the MSE and the gradients change. Look at `gradient_w` in step 0: that is the number we use in the first update.

❓ In step 0 all predictions are 0 and the true values are positive. Is the gradient of $w$ positive or negative? Why therefore does $w$ *increase*?

❓ Which values are $w$ and $b$ approaching? Compare with the true line $y=2x+1$.

In [ ]:
x_small = np.array([0.0, 1.0, 2.0, 3.0])
y_small = np.array([1.0, 3.0, 5.0, 7.0])
alpha = 0.05
w, b = 0.0, 0.0

def mse(y_true, y_pred):
    return np.mean((y_true - y_pred) ** 2)

for step in range(6):
    err = (w * x_small + b) - y_small              # prediction - truth, for each point
    grad_w = 2 / len(x_small) * np.sum(err * x_small)
    grad_b = 2 / len(x_small) * np.sum(err)
    print(f"step {step}: w = {w:.3f}  b = {b:.3f}  MSE = {mse(y_small, w * x_small + b):.3f}  gradient_w = {grad_w:.3f}  gradient_b = {grad_b:.3f}")
    w -= alpha * grad_w
    b -= alpha * grad_b

## 3. Linear Regression from Scratch

We begin with synthetic data containing one input feature. Using one feature makes it easy to visualize the data now and the fitted regression line later, after training.

In [ ]:
X_reg, y_reg = make_regression(
    n_samples=100,
    n_features=1,
    noise=20,
    random_state=42,
)

plt.figure(figsize=(7, 4))
plt.scatter(X_reg, y_reg, alpha=0.7)
plt.xlabel("Feature x")
plt.ylabel("Target y")
plt.title("Synthetic Regression Data")
plt.show()

### Guided Implementation

The prediction is $\hat{y}_i = wx_i + b$ and the loss is the Mean Squared Error. The gradients are the same formulas as in the hand calculation above.

Complete the four marked parts together during the exercise:

1. predictions,
2. error and loss,
3. gradients,
4. parameter updates.

In [ ]:
class LinearRegressionGD:

    def __init__(self, learning_rate=0.05, epochs=100):
        self.learning_rate = learning_rate
        self.epochs = epochs
        self.weight = 0.0
        self.bias = 0.0
        self.loss_history = []

    def fit(self, X, y):
        X = np.asarray(X).ravel()
        y = np.asarray(y)
        n = len(y)

        for _ in range(self.epochs):
            # 1. Make predictions
            y_pred = ...

            # 2. Calculate the error (prediction - y) and MSE
            error = ...
            loss = ...
            self.loss_history.append(loss)

            # 3. Calculate the gradients
            dw = ...
            db = ...

            # 4. Update the parameters
            self.weight = ...
            self.bias = ...

        return self

    def predict(self, X):
        X = np.asarray(X).ravel()
        return self.weight * X + self.bias

### Train the Model and Check Convergence

In [ ]:
gd_model = LinearRegressionGD(learning_rate=0.05, epochs=100)
gd_model.fit(X_reg, y_reg)

print("Weight:", round(gd_model.weight, 3))
print("Bias:", round(gd_model.bias, 3))
print("Final MSE:", round(gd_model.loss_history[-1], 3))

In [ ]:
# Plot gd_model.loss_history (x: epoch, y: MSE).

❓ Does the loss decrease during training? Does it level off — what does that suggest?

In [ ]:
order = np.argsort(X_reg.ravel())

plt.figure(figsize=(7, 4))
plt.scatter(X_reg, y_reg, alpha=0.7, label="Observations")
plt.plot(
    X_reg.ravel()[order],
    gd_model.predict(X_reg)[order],
    color="red",
    linewidth=2,
    label="Gradient descent",
)
plt.xlabel("Feature x")
plt.ylabel("Target y")
plt.title("Linear Regression Fitted by Gradient Descent")
plt.legend()
plt.show()

### Compare with Scikit-learn

In [ ]:
# Create and fit a scikit-learn LinearRegression model.
# Print its coefficient and intercept next to gd_model.weight and gd_model.bias.

❓ Are the parameters exactly identical? If not, why might they differ slightly?

### 🧩 Independent Task 1: Investigate the Learning Rate

Train the model using the learning rates below and plot all loss curves in one figure (logarithmic y-axis).

- Which learning rate converges slowly?
- Which learning rate converges successfully?
- Does any learning rate produce unstable training?

In [ ]:
learning_rates = [0.001, 0.05, 1.0]

plt.figure(figsize=(8, 5))

for learning_rate in learning_rates:
    # Create and train a model.
    # Plot its loss history and add an informative label.
    pass

plt.xlabel("Epoch")
plt.ylabel("MSE")
plt.yscale("log")
plt.title("Effect of the Learning Rate")
plt.legend()
plt.show()

### 🧩 Independent Task 2: Linear Regression on Real Data

The built-in **diabetes** dataset: the target is a measure of disease progression one year after baseline. We use one feature, `bmi`, so we can reuse `LinearRegressionGD`.

Gradient descent works best with scaled data, so we standardize both `X` and `y` (using **training** statistics only). The model then predicts in *standardized units*.

**Your job:** convert the standardized predictions back to the original units (inverse of $(y-\bar y)/s_y$) and compare with scikit-learn using MAE and RMSE on the test set.

In [ ]:
diab = load_diabetes(as_frame=True)
X_d = diab.data[["bmi"]].to_numpy()
y_d = diab.target.to_numpy()
X_d_train, X_d_test, y_d_train, y_d_test = train_test_split(X_d, y_d, test_size=0.25, random_state=42)

x_scaler = StandardScaler().fit(X_d_train)
y_mean, y_std = y_d_train.mean(), y_d_train.std()

gd_d = LinearRegressionGD(learning_rate=0.1, epochs=300)
gd_d.fit(x_scaler.transform(X_d_train), (y_d_train - y_mean) / y_std)

pred_scaled = gd_d.predict(x_scaler.transform(X_d_test))
pred_gd = ...      # TODO: convert back to original units using y_mean and y_std

sk_d = LinearRegression().fit(X_d_train, y_d_train)
pred_sk = sk_d.predict(X_d_test)

# TODO: print MAE and RMSE (on y_d_test) for both models

❓ Are the two models similar? What does that tell us about our implementation?

❓ Does similarity to scikit-learn prove that a one-feature linear model is *good* for this problem? (Compare the RMSE with the spread of the target, `y_d_test.std()`.)

❓ Why did we fit the scalers on the training data only?

## 4. From Linear to Logistic Regression

Linear regression predicts an unrestricted numerical value. Logistic regression estimates the probability that an observation belongs to class 1.

First, a linear combination is calculated, $z = Xw + b$. The **sigmoid function** converts it to a value between 0 and 1:

$$
\sigma(z)=\frac{1}{1+e^{-z}},\qquad P(y=1\mid X)=\sigma(Xw+b)
$$

❓ Why is ordinary linear regression unsuitable for predicting probabilities?

### The Sigmoid Function

Complete the function and inspect its shape. (Hint: `np.exp`.)

In [ ]:
def sigmoid(z):
    return ...


z = np.linspace(-10, 10, 200)

plt.figure(figsize=(7, 4))
plt.plot(z, sigmoid(z))
plt.axhline(0.5, color="grey", linestyle="--")
plt.axvline(0, color="grey", linestyle="--")
plt.xlabel("z")
plt.ylabel("sigmoid(z)")
plt.title("Sigmoid Function")
plt.show()

### Logistic Loss and Gradient Descent

Logistic regression is trained using **binary cross-entropy**:

$$
L = -\frac{1}{n}\sum_{i=1}^{n}\left[y_i\log(p_i)+(1-y_i)\log(1-p_i)\right]
$$

*Intuition:* if the true class is 1 and the model says $p=0.01$, the loss is $-\log 0.01\approx 4.6$ — a confident mistake is punished heavily. If $p=0.99$, the loss is almost 0.

The gradient has the same shape as in linear regression: $\frac1n X^\top(p-y)$. The optimization pattern is therefore identical:

1. calculate predictions, 2. calculate the loss and gradients, 3. update the parameters, 4. repeat.

The implementation below is provided so that we can focus on the shared pattern and on classifier evaluation.

In [ ]:
class LogisticRegressionGD:

    def __init__(self, learning_rate=0.1, epochs=2000):
        self.learning_rate = learning_rate
        self.epochs = epochs
        self.weights = None
        self.bias = 0.0
        self.loss_history = []

    def fit(self, X, y):
        X = np.asarray(X)
        y = np.asarray(y)
        n, n_features = X.shape

        self.weights = np.zeros(n_features)

        for _ in range(self.epochs):
            z = X @ self.weights + self.bias
            probabilities = sigmoid(z)

            epsilon = 1e-12
            loss = -np.mean(
                y * np.log(probabilities + epsilon)
                + (1 - y) * np.log(1 - probabilities + epsilon)
            )
            self.loss_history.append(loss)

            error = probabilities - y
            dw = (X.T @ error) / n
            db = np.mean(error)

            self.weights -= self.learning_rate * dw
            self.bias -= self.learning_rate * db

        return self

    def predict_proba(self, X):
        X = np.asarray(X)
        return sigmoid(X @ self.weights + self.bias)

    def predict(self, X, threshold=0.5):
        return (self.predict_proba(X) >= threshold).astype(int)

❓ Identify the four repeated steps in the `fit()` method. What is different compared with the linear-regression implementation?

❓ Why does `predict()` require a threshold? Is the threshold part of training?

## 5. Binary Classification: Breast Cancer Data

We use the Breast Cancer Wisconsin dataset included in scikit-learn. Each row is one tumour sample described by 30 numerical measurements computed from a digitized image.

- `1` = malignant tumour,
- `0` = benign tumour.

In [ ]:
cancer = load_breast_cancer(as_frame=True)

X_cls = cancer.data.copy()
y_cls = (cancer.target == 0).astype(int)
y_cls.name = "malignant"

print("Feature matrix shape:", X_cls.shape)
print("Class counts:")
print(y_cls.value_counts().rename(index={0: "benign", 1: "malignant"}))

### Train–Test Split and Scaling

The scaler is fitted only on the training data, so that no information from the test set influences training.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_cls,
    y_cls,
    test_size=0.2,
    random_state=42,
    stratify=y_cls,
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Training observations:", len(X_train))
print("Test observations:", len(X_test))

❓ Why `fit_transform()` on the training set but only `transform()` on the test set?

❓ Why is scaling important for gradient descent?

In [ ]:
logistic_gd = LogisticRegressionGD(learning_rate=0.1, epochs=2000)
logistic_gd.fit(X_train_scaled, y_train)

y_pred_gd = logistic_gd.predict(X_test_scaled)

plt.figure(figsize=(7, 4))
plt.plot(logistic_gd.loss_history)
plt.xlabel("Epoch")
plt.ylabel("Binary cross-entropy")
plt.title("Logistic Regression Training Loss")
plt.show()

## 6. Classification Evaluation

A confusion matrix distinguishes four outcomes:

| | predicted malignant | predicted benign |
|---|---|---|
| **actually malignant** | TP (true positive) | FN (false negative) |
| **actually benign** | FP (false positive) | TN (true negative) |

$$
Accuracy = \frac{TP+TN}{TP+TN+FP+FN},\quad
Precision = \frac{TP}{TP+FP},\quad
Recall = \frac{TP}{TP+FN}
$$

* **Precision:** when the model raises an alarm ("malignant"), how often is it right?
* **Recall:** of all truly malignant tumours, how many did the model find?
* **F1-score:** $F_1 = 2\cdot\frac{Precision\cdot Recall}{Precision+Recall}$ — the *harmonic* mean of precision and recall. Unlike the ordinary average it is dominated by the weaker of the two. Example: precision = 1.0, recall = 0.01 → average 0.505 (looks fine), but $F_1\approx 0.02$ (correctly looks terrible). A model cannot hide a very bad recall behind a perfect precision.

**Why accuracy alone is not enough:** imagine 95 benign and 5 malignant tumours and a "model" that always says *benign*. Accuracy 95 %, recall 0 %. Always look at the class distribution and at recall/precision for the class you care about.

❓ Which error is especially dangerous in this context, and why?

### 🧩 Independent Task 3: Evaluate the Classifier

1. Display the confusion matrix.
2. Calculate accuracy, precision, recall, and F1-score.
3. Interpret the false negatives in the context of the problem.
4. Decide whether accuracy alone provides sufficient information.

Remember: the positive class (`1`) represents a malignant tumour.

In [ ]:
# Display the confusion matrix for y_test and y_pred_gd.

In [ ]:
# Calculate accuracy, precision, recall, and F1-score for y_test and y_pred_gd.

### Compare with Scikit-learn

Train `sklearn.linear_model.LogisticRegression` on the same training data and evaluate it on the same test data.

In [ ]:
# Create and fit LogisticRegression(max_iter=2000).
# Generate predictions for X_test_scaled.
# Calculate the same four metrics.

❓ Do the two implementations produce similar results? (scikit-learn adds L2 regularization by default.)

❓ Why should both models be evaluated on exactly the same test observations?

## 7. Threshold, ROC Curve and AUC

Logistic regression outputs a **probability**. The class prediction depends on the **threshold** $t$: predict "malignant" if $p\ge t$.

* **Lower threshold** → more alarms → higher recall, lower precision (more false positives).
* **Higher threshold** → fewer alarms → higher precision, lower recall (more false negatives).

Changing the threshold does **not** retrain the model — only the decision rule changes.

### ROC curve

For every possible threshold we compute one point:

* **TPR** (true positive rate) = recall = $\frac{TP}{TP+FN}$ — the share of sick patients we catch,
* **FPR** (false positive rate) = $\frac{FP}{FP+TN}$ — the share of healthy patients we wrongly alarm.

The ROC curve connects these points. Threshold 1 → we never alarm → point (0, 0). Threshold 0 → we always alarm → point (1, 1). The diagonal is random guessing; the closer the curve is to the top-left corner, the better.

**AUC** (area under the ROC curve): 0.5 = random, 1.0 = perfect. Intuitively, AUC is the probability that a **randomly chosen positive sample receives a higher predicted probability than a randomly chosen negative one**. It evaluates how well the model *ranks* samples and does **not depend on the threshold**.

### Drawing ROC by hand (6 patients)

Sort by predicted probability (highest first). Walk down the list: positive → one step **up**, negative → one step **right** (steps of 1/3, because there are 3 positives and 3 negatives).

| probability | 0.9 | 0.8 | 0.7 | 0.6 | 0.4 | 0.2 |
|---|---|---|---|---|---|---|
| true class | + | + | − | + | − | − |

Count the (positive, negative) pairs where the positive has the higher probability: 9 pairs in total → AUC = (number of correctly ordered pairs)/9. Verify below.

In [ ]:
y_toy = np.array([1, 1, 0, 1, 0, 0])
p_toy = np.array([0.9, 0.8, 0.7, 0.6, 0.4, 0.2])
print("AUC of the toy example:", roc_auc_score(y_toy, p_toy))
fpr_t, tpr_t, thr_t = roc_curve(y_toy, p_toy)
plt.figure(figsize=(4, 4)); plt.step(fpr_t, tpr_t, where="post", marker="o")
plt.plot([0, 1], [0, 1], "k--"); plt.xlabel("FPR"); plt.ylabel("TPR"); plt.title("Toy ROC"); plt.show()

Now the real ROC curve of our model. The red points show where the thresholds 0.1, 0.5 and 0.9 lie on the curve:

In [ ]:
p_test = logistic_gd.predict_proba(X_test_scaled)

fpr, tpr, _ = roc_curve(y_test, p_test)
auc = roc_auc_score(y_test, p_test)

plt.figure(figsize=(5.5, 5))
plt.plot(fpr, tpr, label=f"logistic regression (AUC = {auc:.3f})")
plt.plot([0, 1], [0, 1], "k--", label="random guessing")

for t in [0.1, 0.5, 0.9]:
    tn, fp, fn, tp = confusion_matrix(y_test, (p_test >= t).astype(int)).ravel()
    plt.plot(fp / (fp + tn), tp / (tp + fn), "ro")
    plt.annotate(f"t={t}", (fp / (fp + tn), tp / (tp + fn)), textcoords="offset points", xytext=(8, -12))

plt.xlabel("False positive rate (1 - specificity)")
plt.ylabel("True positive rate (recall)")
plt.title("ROC curve")
plt.legend(loc="lower right")
plt.show()

### 🧩 Independent Task 4: Choose a Threshold

A doctor says: *"Missing a malignant tumour is unacceptable. I need recall of at least 0.98."*

1. For thresholds `np.linspace(0.01, 0.99, 99)` compute precision and recall on the test set.
2. Find the **highest** threshold that still gives recall ≥ 0.98.
3. Report precision and the confusion matrix for it. What did we pay for the higher recall?

In [ ]:
thresholds = np.linspace(0.01, 0.99, 99)

# TODO: loop over the thresholds, compute recall and precision of (p_test >= t),
#       store them, and find the highest threshold with recall >= 0.98
...

❓ Is there a universally best threshold? What information do you need to choose one?

❓ AUC is the same for every threshold. Why?

## 8. Exit Ticket

Answer briefly in your own words:

1. What does gradient descent change during training?
2. What is the role of the learning rate? Is it the same thing as the decision threshold?
3. What is the main difference between linear and logistic regression outputs?
4. Why can accuracy be misleading when evaluating a classifier?
5. What does a point on the ROC curve represent and what does AUC = 0.5 mean?
6. When would you prefer a lower threshold, and what error increases?